# 04 — Quantum Advantage & Oracles

This is the notebook that answers *why quantum can be faster at all*.

A hidden function returns a single **parity bit** from several inputs. Classically there's no
shortcut: you query it once per input bit, tallying results as you go. Quantumly, you put the
input in **superposition**, query the oracle **once**, and **phase kickback + interference**
reveal the same answer in a single query. This is the **Bernstein–Vazirani algorithm** — the
exact oracle-query idea that Grover generalizes to search in notebook 05.

Maps to the `/demo` deck slides "Classical parity check" / "Quantum parity check"
(`ParityAnimation`).

> **Running this notebook** — from the repo root:
> ```bash
> uv run --with jupyterlab --with qiskit==1.2.0 --with matplotlib --with pylatexenc --with httpx \
>   jupyter lab notebooks/04-quantum-advantage-oracles.ipynb
> ```
> The optional "Ask Bob" step needs `cd mcp-server && uv run server.py --http` in another
> terminal — it degrades gracefully if that isn't running.

## Step 0 — Install dependencies

In [ ]:
%pip install -q qiskit==1.2.0 matplotlib pylatexenc httpx
print("✅ Qiskit + matplotlib + pylatexenc + httpx ready")

## Step 0b — Load the workshop helpers (`notebooks/lib/`)

In [ ]:
import sys
from pathlib import Path

_notebooks_dir = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd() / "notebooks"
if str(_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(_notebooks_dir))

from lib.visuals import draw_circuit, plot_counts
from lib.backend import explain_with_bob

## Step 1 — The hidden function

`SECRET` is a bitstring **unknown to the querying logic below** — both the classical and
quantum sides only get to call `oracle(x)`, never read `SECRET` directly. The oracle computes
the parity of `x AND SECRET` (bitwise AND, then XOR all the resulting bits):

$$f(x) = \bigoplus_i (x_i \cdot s_i)$$

In [ ]:
SECRET = "1011"  # hidden bitstring — pretend you don't know this yet
n = len(SECRET)


def oracle(x_bits: str) -> int:
    """The hidden function: parity of x AND SECRET (bitwise)."""
    return sum(int(x) & int(s) for x, s in zip(x_bits, SECRET)) % 2

## Step 2 — Classical: one query per bit

Querying the oracle at the unit vector `e_i` (all zeros except a `1` at position `i`) reveals
exactly `SECRET[i]` — because the AND then picks out only that one bit. Recovering the full
secret takes `n` queries, tallying one bit per query.

In [ ]:
classical_queries = 0
recovered_bits = []
for i in range(n):
    x = ["0"] * n
    x[i] = "1"
    bit = oracle("".join(x))
    classical_queries += 1
    recovered_bits.append(str(bit))
    print(f"Query {classical_queries}: f({''.join(x)}) = {bit}")

recovered_classical = "".join(recovered_bits)
print(f"\nRecovered secret classically: {recovered_classical} in {classical_queries} queries")

## Step 3 — Quantum: one query, total

Put an ancilla qubit in `|-⟩` (via `X` then `H`), superpose the `n` data qubits with `H`, then
query the oracle **once** as a single unitary: a `CX` from data qubit `i` to the ancilla for
every bit where `SECRET[i] == 1`. Because the ancilla is in `|-⟩`, each `CX` **kicks the phase
back** onto the data qubit instead of flipping the ancilla. A final layer of `H` gates turns
that phase pattern directly into `SECRET`.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(n + 1, n)
qc.x(n)              # ancilla → |1⟩
qc.h(range(n + 1))    # superpose everything; ancilla is now |-⟩

for i, bit in enumerate(SECRET):
    if bit == "1":
        qc.cx(i, n)   # phase kickback onto qubit i whenever SECRET[i] == 1

qc.h(range(n))
qc.measure(range(n), range(n))

draw_circuit(qc, title="Bernstein\u2013Vazirani oracle circuit (1 query)")

## Step 3b — Ask Bob to explain the oracle

In IBM Bob's chat, try asking Bob to explain this exact circuit — or run the cell below, which
calls the same MCP `explain_circuit` tool directly (skips gracefully if the MCP server isn't
running in `--http` mode).

In [ ]:
bv_code = """from qiskit import QuantumCircuit

n = 4
qc = QuantumCircuit(n + 1, n)
qc.x(n)
qc.h(range(n + 1))
qc.cx(0, n); qc.cx(2, n); qc.cx(3, n)  # phase kickback for SECRET = 1011
qc.h(range(n))
qc.measure(range(n), range(n))
"""

explanation = explain_with_bob(bv_code)
if explanation:
    print(explanation)
else:
    print(
        "MCP server not reachable at http://localhost:8765 \u2014 start it with "
        "`cd mcp-server && uv run server.py --http`, or ask Bob directly in "
        "your assistant chat."
    )

## Step 4 — Run it: one query, one answer

In [ ]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()
job = sampler.run([(qc, None, 1024)])
result = job.result()
counts = dict(result[0].data.c.get_counts())
plot_counts(counts, title="Bernstein\u2013Vazirani result (1 query)")

# Qiskit's classical register is little-endian (qubit 0 = rightmost char) — reverse to
# read the secret bits back in the same left-to-right order we defined SECRET in.
recovered_quantum = max(counts, key=counts.get)[::-1]
print(f"Recovered secret quantumly: {recovered_quantum} in 1 query")

## Step 5 — Read it: same secret, radically fewer queries

In [ ]:
print(f"Classical: {recovered_classical} in {classical_queries} queries")
print(f"Quantum:   {recovered_quantum} in 1 query")
assert recovered_classical == recovered_quantum == SECRET
print("\u2705 Both sides agree — and the quantum side needed 1 query instead of", classical_queries)

## What's next?

The oracle-query idea here — mark information in *phase*, then use interference to read it out
in one shot — is exactly what **Grover's algorithm** generalizes to *search* in
**notebook 05, the capstone**: instead of revealing a hidden bitstring, the oracle marks the
satisfying assignment of a boolean formula, and Grover *amplifies* it over repeated queries.

---
*Quantum Experience Lab — IBM Bob + Qiskit workshop*